In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')

In [3]:
import os
import math
import time
import warnings
import datetime
from pathlib import Path
from typing import Any, Callable, Optional, Union, cast

import torch
import torch.nn as nn

import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2
from torch.utils.data import DataLoader, default_collate

import numpy as np

%load_ext autoreload
%autoreload 2

from computer_vision.torch_video.data.sampler import RandomClipSampler, UniformClipSampler
from computer_vision.torch_video.data.dataset import UCF101, ConvertTCHWtoCTHW, DATA_MEAN, DATA_STD, NUM_CLASSES
from computer_vision.torch_video.parameter_parser import parser

from computer_vision.torch_video.utils.progress import MetricLogger, form_stats, save_metrics
from computer_vision.torch_video.utils.train_utils import evaluate, train_one_epoch
from computer_vision.torch_video.utils.torch_utils import init_seeds, seed_worker, init_distributed_mode, initialize_weights,\
reduce_across_processes, save_checkpoint
from computer_vision.torch_video.utils.plotting import plot_results

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import os
import bisect
from pathlib import Path
from typing import Any, Callable, Optional, Union, cast

import numpy as np

import torch
import torch.nn as nn
from torchvision.transforms import v2
from torchcodec.decoders import VideoDecoder, AudioDecoder
from torchcodec.samplers import clips_at_regular_timestamps, clips_at_random_timestamps

from computer_vision.torch_video.data.base import VisionDataset
from computer_vision.torch_video.data.utils import find_classes, has_file_allowed_extension, make_dataset, compute_clip_start_times
from computer_vision.torch_video.data.dataset import VideoClipMetadata

DATA_MEAN=(0.43216, 0.394666, 0.37645)
DATA_STD=(0.22803, 0.22145, 0.216989)
NUM_CLASSES=101

In [5]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask'
output_dirpath=Path('D:/results/ucf101/train')
metadata_path=data_dirpath/'metadata.pt'
class_id_path=annotation_path/"classInd.txt"
allowed_time=10/60

arguments= f"""-d {root} -a {annotation_path} -m {metadata_path} -o {output_dirpath} -c {class_id_path}
--data-fold 1 --frame-rate 4 --clip-duration 2 --step-duration 1.7 
--device cpu --time {allowed_time} --n-batches 4 --epochs 2
""" #--use-cutmix-mixup 
args=parser.parse_args(arguments.split())

In [7]:
class UCF101(VisionDataset):
    def __init__(self, root: Union[str, Path], annotation_path: Union[str, Path], frame_rate:float=8, clip_duration:float=2, step_duration:float=1.7,
                 train:bool=True, fold:int=None, sampling_type:str='regular', use_audio:bool=False, transforms:Optional[Callable]=None, 
                 decoder_transforms:Optional[list[Callable]]=None, metadata_path:str=None, num_ffmpeg_threads:int=0 )->None:

        if not 1<=fold<=3: raise ValueError(f"Fold should be between 1 and 3, but got {fold}")
            
        super().__init__(root=root, transforms=transforms)
        
        extension=("avi",)
        self.train=train
        self.frame_rate=frame_rate
        self.clip_duration=clip_duration
        self.step_duration=step_duration
        self.classes, class_to_idx=find_classes(self.root)
        self.idx_to_class={v:k for k, v in class_to_idx.items()}
        self.samples=make_dataset(self.root, class_to_idx, extension, is_valid_file=None)

        # We bookkeep the full version of video clip metadata because we want to be able to return the metadata of full version rather than the
        # subset version of video clips
        video_list=[x[0] for x in self.samples]
        self.full_video_clip_metadata=VideoClipMetadata(video_paths=video_list,clip_length_in_seconds=clip_duration, clip_stride_in_seconds=step_duration,
                                              frame_rate=frame_rate, sampling_type=sampling_type, use_audio=use_audio, metadata_path=metadata_path,
                                              num_ffmpeg_threads=num_ffmpeg_threads)
        self.invalid_video=[self.full_video_clip_metadata.video_paths[i] for i in self.full_video_clip_metadata.invalid_video]
        self.indices=self._select_fold(self.full_video_clip_metadata.video_paths, self.invalid_video, 
                                       annotation_path, fold, train)
        self.video_clip_metadata=self.full_video_clip_metadata.subset(self.indices)
        self.transforms=transforms
        self.decoder_transforms=decoder_transforms
        
    @property
    def metadata(self)->dict[str, Any]:
        """Return video clip metadata of the whole dataset"""
        return self.video_clip_metadata.metadata

    def _select_fold(self, video_list:list[str], invalid_video:list[str], annotation_path:str, fold:int, train:bool)->list[int]:
        """Read txt file listing video files for the specified fold and find the indices of those files in all `video_list`
        Args:
            video_list (list[str]): List of all absolute paths to all video files
            invalid_video (list[int]): List of paths to videos that cannot be used since video time < required clip time
            annotation_path (str): Path to directory containing annotation txt file for each fold
            fold (int): Data fold with options of 1, 2 or 3
            train (bool): Whether data is for training or testing
        Returns:
            (list[int]): Indices of selected video from video_list
        """
        name='train' if train else 'test'
        name=f"{name}list{fold:02d}.txt"
        f=os.path.join(annotation_path, name)
        selected_files=set()
        with open(f) as fid:
            data=fid.readlines()
            data=[x.strip().split(" ")[0] for x in data]
            data=[os.path.join(self.root, *x.split("/")) for x in data]
            selected_files.update(data)
        indices=[i for i in range(len(video_list)) if (video_list[i] in selected_files and video_list[i] not in invalid_video)]
        return indices
        
    def __len__(self)->int: return self.video_clip_metadata.num_clips()

    def __getitem__(self, idx:int)->tuple[torch.Tensor, torch.Tensor, dict[str,Any], int]:
        """Get input video and target label"""
        
        video, audio, info, video_idx=self.video_clip_metadata.get_clip(idx, transforms=self.decoder_transforms)
        label=self.samples[self.indices[video_idx]][1]
        info['class']=self.idx_to_class[label]
        if self.transforms is not None: video=self.transforms(video)
        if self.video_clip_metadata.use_audio: return video, audio, label, video_idx, info
        return video, label, video_idx, info
        
# Data loader
train_dataset=UCF101(root=args.data_path, annotation_path=args.annotation_path, frame_rate=args.frame_rate, 
                     clip_duration=args.clip_duration, step_duration=args.step_duration,
                     train=True, metadata_path=args.metadata_path, fold=args.data_fold, sampling_type='random', use_audio=False,
                     decoder_transforms=[v2.RandomCrop(size=args.train_crop_size),
                                         v2.Resize(args.train_resize_size)],
                     transforms=v2.Compose([
                         v2.RandomHorizontalFlip(p=args.hflip_prob)])
                    ) 


In [ ]:
print(f"{len(train_dataset.full_video_clip_metadata.video_paths)=}, {len(train_dataset.samples)=}")
print(f"{len(train_dataset.full_video_clip_metadata.invalid_video)=}, {train_dataset.full_video_clip_metadata.invalid_video=}")
print(f"\n{sum([1 for x in train_dataset.full_video_clip_metadata.clip_start_times if x is None])}")
print(f"{sum([1 for x in train_dataset.full_video_clip_metadata.num_clips_per_video if x==0])}")
len(train_dataset.full_video_clip_metadata.num_clips_per_video), len(train_dataset.full_video_clip_metadata.clip_start_times)

In [16]:
video_meta=train_dataset.video_clip_metadata
for idx in range(len(train_dataset)):
    video_idx, clip_idx=video_meta.get_clip_location(idx)
    video_path=video_meta.video_paths[video_idx]
    fpath, label=train_dataset.samples[train_dataset.indices[video_idx]]
    assert fpath==video_path
    class_name=train_dataset.idx_to_class[label]
    assert class_name in video_path